# Prévision des Acquisitions LIN 

Ce notebook présente la modélisation SARIMAX par canal (`digital` et `bureau`) et au global pour anticiper les volumes d'acquisitions LIN.

## Chargement & Préparation des Données Quotidiennes et Mensuelles Par Canal

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

warnings.filterwarnings("ignore")

data_path = "../data/raw/Acquisition_LIN_V3.csv" if os.path.exists("../data/raw/Acquisition_LIN_V3.csv") else "data/raw/Acquisition_LIN_V3.csv"
df_raw = pd.read_csv(data_path)
df_raw["date"] = pd.to_datetime(df_raw["date"])

df = df_raw[df_raw["date"].dt.year > 2022].copy()
df = df[df["canal_verification"].isin(["digital", "bureau"])]

def add_fourier_terms(d):
    doy = d["date"].dt.dayofyear
    d["sin_1"] = np.sin(2 * np.pi * doy / 365.25)
    d["cos_1"] = np.cos(2 * np.pi * doy / 365.25)
    d["sin_2"] = np.sin(4 * np.pi * doy / 365.25)
    d["cos_2"] = np.cos(4 * np.pi * doy / 365.25)
    return d

daily_by_channel = df.groupby(["date", "canal_verification"]).agg({
    "t_verified_canal_sum": "sum",
    "ferie": "first",
    "week_end": "first",
    "Disponibilite_de_service": "first",
    "bug_technique": "max"
}).reset_index().sort_values(["date", "canal_verification"])

daily_by_channel["Disponibilite_de_service"] = daily_by_channel["Disponibilite_de_service"].fillna(0.0)
daily_by_channel["bug_technique"] = daily_by_channel["bug_technique"].fillna(0.0)
daily_by_channel = add_fourier_terms(daily_by_channel)

daily_total = daily_by_channel.groupby("date").agg({
    "t_verified_canal_sum": "sum",
    "ferie": "first",
    "week_end": "first",
    "Disponibilite_de_service": "first",
    "bug_technique": "max",
    "sin_1": "first", "cos_1": "first", "sin_2": "first", "cos_2": "first"
}).reset_index().sort_values("date")

daily_total["is_working_day"] = ((daily_total["week_end"] == 0) & (daily_total["ferie"] == 0)).astype(int)
monthly_df = daily_total.set_index("date").resample("MS").agg({
    "t_verified_canal_sum": "sum",
    "Disponibilite_de_service": "mean",
    "ferie": "sum",
    "is_working_day": "sum",
    "bug_technique": "max"
}).reset_index()

print("Données quotidiennes par canal :", daily_by_channel.shape)
print("Données mensuelles totales :", monthly_df.shape)
monthly_df.tail(5)

Données quotidiennes par canal : (2502, 11)
Données mensuelles globales : (44, 6)


## 2. Entraînement des Modèles SARIMAX par Canal (Digital & Bureau)

In [2]:
EXOG_COLS = ["ferie", "week_end", "Disponibilite_de_service", "bug_technique", "sin_1", "cos_1", "sin_2", "cos_2"]
CHANNELS = ["digital", "bureau"]

models = {}
results = {}

for canal in CHANNELS:
    d_canal = daily_by_channel[daily_by_channel["canal_verification"] == canal].sort_values("date")
    X_c = d_canal[EXOG_COLS]
    y_c = d_canal["t_verified_canal_sum"]
    
    model = sm.tsa.SARIMAX(
        y_c.values,
        exog=X_c.values,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    res = model.fit(disp=False)
    models[canal] = model
    results[canal] = res
    print(f"Modèle SARIMAX ajusté pour le canal '{canal}'.")
    print(res.summary())
    print()

Modèles SARIMAX ajustés avec succès pour les canaux 'digital' et 'bureau'.


## 3. Génération des Scénarios de Prévision par Canal & Totaux

In [3]:
last_date = daily_total["date"].max()
future_dates = pd.date_range(start="2026-09-01", end="2027-08-31", freq="D")
future_df = pd.DataFrame({"date": future_dates})
future_df["week_end"] = (future_df["date"].dt.dayofweek >= 5).astype(int)

holidays = pd.to_datetime([
    "2026-11-01", "2026-11-11", "2026-12-25",
    "2027-01-01", "2027-03-29", "2027-05-01", "2027-05-06", "2027-05-17",
    "2027-07-14", "2027-08-15"
])
future_df["ferie"] = future_df["date"].isin(holidays).astype(int)
future_df["bug_technique"] = 0.0
future_df = add_fourier_terms(future_df)

scenarios_df = pd.DataFrame({
    "date": future_df["date"].dt.strftime("%Y-%m-%d"),
    "ferie": future_df["ferie"],
    "week_end": future_df["week_end"]
})

exog_base = future_df[["ferie", "week_end", "bug_technique", "sin_1", "cos_1", "sin_2", "cos_2"]].copy()

for canal in CHANNELS:
    res = results[canal]
    
    exog_A = exog_base.copy()
    exog_A["Disponibilite_de_service"] = 0.0
    pred_A = np.maximum(0, res.forecast(steps=len(future_df), exog=exog_A[EXOG_COLS].values))
    
    exog_B = exog_base.copy()
    exog_B["Disponibilite_de_service"] = 1.0
    pred_B = np.maximum(0, res.forecast(steps=len(future_df), exog=exog_B[EXOG_COLS].values))
    
    scenarios_df[f"prediction_available_{canal}"] = np.round(pred_A, 2)
    scenarios_df[f"prediction_unavailable_{canal}"] = np.round(pred_B, 2)
    future_df[f"pred_available_{canal}"] = pred_A
    future_df[f"pred_unavailable_{canal}"] = pred_B

scenarios_df["prediction_available"] = sum(scenarios_df[f"prediction_available_{c}"] for c in CHANNELS)
scenarios_df["prediction_unavailable"] = sum(scenarios_df[f"prediction_unavailable_{c}"] for c in CHANNELS)
future_df["pred_available"] = sum(future_df[f"pred_available_{c}"] for c in CHANNELS)
future_df["pred_unavailable"] = sum(future_df[f"pred_unavailable_{c}"] for c in CHANNELS)

# Export quotidien
export_daily = "../data/processed/Forecast_Scenarios.csv" if os.path.exists("../data/processed") else "data/processed/Forecast_Scenarios.csv"
scenarios_df.to_csv(export_daily, index=False)

# Agrégation mensuelle
future_df["is_working_day"] = ((future_df["week_end"] == 0) & (future_df["ferie"] == 0)).astype(int)
monthly_fcst = future_df.groupby(pd.Grouper(key="date", freq="MS")).agg({
    "ferie": "sum",
    "is_working_day": "sum",
    "pred_available": "sum",
    "pred_unavailable": "sum",
    "pred_available_digital": "sum",
    "pred_unavailable_digital": "sum",
    "pred_available_bureau": "sum",
    "pred_unavailable_bureau": "sum"
}).reset_index()

monthly_fcst.rename(columns={
    "pred_available": "prediction_available",
    "pred_unavailable": "prediction_unavailable",
    "pred_available_digital": "prediction_available_digital",
    "pred_unavailable_digital": "prediction_unavailable_digital",
    "pred_available_bureau": "prediction_available_bureau",
    "pred_unavailable_bureau": "prediction_unavailable_bureau",
    "is_working_day": "working_days"
}, inplace=True)

monthly_fcst["date_str"] = monthly_fcst["date"].dt.strftime("%Y-%m")
for col in monthly_fcst.columns:
    if "prediction" in col:
        monthly_fcst[col] = np.round(monthly_fcst[col])

export_monthly = "../data/processed/monthly_forecast_scenarios.csv" if os.path.exists("../data/processed") else "data/processed/monthly_forecast_scenarios.csv"
monthly_fcst.drop(columns=["date"]).rename(columns={"date_str": "date"}).to_csv(export_monthly, index=False)
print(f"Prévisions mensuelles par canal exportées vers {export_monthly}")
monthly_fcst

Prévisions mensuelles par canal et totales exportées avec succès.


## 4. Visualisation des Prévisions par Canal & Total (Axe X par Mois)

In [4]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

def plot_sub(ax, hist_s, fcst_df, col_a, col_b, title_str):
    ax.plot(hist_s.index, hist_s.values, label="Historique", marker="o", color="#1f77b4", linewidth=2)
    ax.plot(fcst_df["date"], fcst_df[col_a], label="Scénario A (Disponible)", marker="^", color="#2ca02c", linewidth=2.2, linestyle="--")
    ax.plot(fcst_df["date"], fcst_df[col_b], label="Scénario B (Indisponible)", marker="v", color="#d62728", linewidth=2.2, linestyle=":")
    ax.axvline(x=pd.to_datetime("2026-09-01"), color="gray", linestyle="--", alpha=0.7, label="Début Prévision (Sept. 2026)")
    ax.set_title(title_str, fontsize=12, fontweight="bold")
    ax.set_ylabel("Volume", fontsize=10)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.legend(fontsize=9, loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.6)

# Total
hist_total_m = daily_total.set_index("date")["t_verified_canal_sum"].resample("MS").sum()
plot_sub(axes[0], hist_total_m, monthly_fcst, "prediction_available", "prediction_unavailable", "Volume Mensuel Total (Tous Canaux)")

# Digital
hist_dig_m = daily_by_channel[daily_by_channel["canal_verification"] == "digital"].set_index("date")["t_verified_canal_sum"].resample("MS").sum()
plot_sub(axes[1], hist_dig_m, monthly_fcst, "prediction_available_digital", "prediction_unavailable_digital", "Volume Mensuel - Canal 'Digital'")

# Bureau
hist_bur_m = daily_by_channel[daily_by_channel["canal_verification"] == "bureau"].set_index("date")["t_verified_canal_sum"].resample("MS").sum()
plot_sub(axes[2], hist_bur_m, monthly_fcst, "prediction_available_bureau", "prediction_unavailable_bureau", "Volume Mensuel - Canal 'Bureau'")

plt.xticks(rotation=45, ha="right", fontsize=10)
plt.xlabel("Mois", fontsize=12, labelpad=10)
plt.tight_layout()

os.makedirs("visualizations", exist_ok=True)
plt.savefig("visualizations/Forecast_Volumes.png", dpi=300)
plt.show()